In [ ]:
import glob, os, sys, platform, time
cands = glob.glob('/kaggle/input/**/splits.json', recursive=True)
DATA_DIR = os.path.dirname(cands[0]); print('DATA', DATA_DIR, 'CPUS', os.cpu_count(), platform.python_version())
import sklearn, xgboost, numpy, pandas, scipy; print(sklearn.__version__, xgboost.__version__, numpy.__version__, pandas.__version__, scipy.__version__)

In [ ]:
"""Shared data loading, feature preprocessing and split generation."""
import numpy as np
import pandas as pd

DATA = DATA_DIR
PROC_LEVELS = ['CAST', 'ANNEAL', 'WROUGHT', 'POWDER', 'OTHER', 'UNKNOWN']
DROP_PHYS = ['PHYS:Weight Fraction', 'PHYS:Atomic Fraction', 'PHYS:Total weight']


def load(dataset, data_dir=DATA):
    """Return records dataframe (meta) and full feature frame aligned to records."""
    fname = {'HV': 'hv_records.csv', 'YS': 'ys_records.csv', 'GE': 'ext_ge_records.csv'}[dataset]
    rec = pd.read_csv(f"{data_dir}/{fname}")
    F = pd.read_csv(f"{data_dir}/composition_features.csv")
    F = F.drop(columns=[c for c in DROP_PHYS + ['PHYS_FAIL'] if c in F.columns])
    X = rec[['comp_key']].merge(F, on='comp_key', how='left').drop(columns=['comp_key'])
    assert len(X) == len(rec) and X.notna().all().all(), "feature merge failed"
    for p in PROC_LEVELS:
        X[f"PROC:{p}"] = (rec.proc == p).astype(float).values
    if dataset == 'YS':
        X['TEST:T_C'] = rec.test_T.values
        X['TEST:tension'] = (rec.test == 'T').astype(float).values
    for ph in ['FCC', 'BCC', 'other']:
        X[f"PHASE:{ph}"] = (rec.phase3 == ph).astype(float).values
    X.index = rec.rid.values
    return rec, X


def feature_columns(X, blocks):
    """blocks: iterable of prefixes, e.g. ('FRAC', 'PHYS', 'MAG', 'PROC', 'TEST')."""
    return [c for c in X.columns if c.split(':')[0] in blocks]


DEFAULT_BLOCKS = ('FRAC', 'PHYS', 'MAG', 'PROC', 'TEST')


class Prep:
    """Train-fold-only preprocessing: drop constant columns, drop near-duplicate columns
    (|Pearson r| > 0.98, keep first), standardize. Fitted on training rows only."""

    def __init__(self, corr_thresh=0.98, scale=True, clip=True):
        self.corr_thresh = corr_thresh
        self.scale = scale
        self.clip = clip  # clamp features to the training-fold range (guards linear/NN extrapolation)

    def fit(self, X):
        X = np.asarray(X, float)
        sd = X.std(0)
        keep = np.where(sd > 1e-10)[0]
        Xk = X[:, keep]
        C = np.abs(np.corrcoef(Xk, rowvar=False))
        np.fill_diagonal(C, 0)
        drop = set()
        for i in range(C.shape[0]):
            if i in drop:
                continue
            for j in np.where(C[i, i + 1:] > self.corr_thresh)[0] + i + 1:
                drop.add(j)
        self.keep = keep[[i for i in range(len(keep)) if i not in drop]]
        self.mu = X[:, self.keep].mean(0)
        self.sd = X[:, self.keep].std(0)
        self.lo = X[:, self.keep].min(0)
        self.hi = X[:, self.keep].max(0)
        return self

    def transform(self, X):
        X = np.asarray(X, float)[:, self.keep]
        if self.clip:
            X = np.clip(X, self.lo, self.hi)
        return (X - self.mu) / self.sd if self.scale else X


def balanced_group_folds(groups, n_splits, rng):
    """Random assignment of groups to folds, balancing record counts (largest groups first,
    random tie-breaking). Returns fold id per record."""
    groups = np.asarray(groups)
    ug, counts = np.unique(groups, return_counts=True)
    order = rng.permutation(len(ug))
    order = order[np.argsort(-counts[order], kind='stable')]
    load_ = np.zeros(n_splits)
    gfold = {}
    for gi in order:
        f = int(np.argmin(load_ + rng.random(n_splits) * 1e-6))
        gfold[ug[gi]] = f
        load_[f] += counts[gi]
    return np.array([gfold[g] for g in groups])


def make_splits(rec, dataset, n_rep=3, n_splits=5, seed=2026, min_elem=40, max_frac=0.6):
    """Outer evaluation protocols. Returns list of dicts:
    {protocol, rep, fold, label, train_idx, test_idx}."""
    n = len(rec)
    out = []
    for r in range(n_rep):
        rng = np.random.default_rng(seed + r)
        perm_fold = np.empty(n, int)
        perm_fold[rng.permutation(n)] = np.arange(n) % n_splits
        for proto, fold_id in [('random', perm_fold),
                               ('composition', balanced_group_folds(rec.comp_key.values, n_splits, rng)),
                               ('system', balanced_group_folds(rec.sys_key.values, n_splits, rng))]:
            for f in range(n_splits):
                te = np.where(fold_id == f)[0]
                tr = np.where(fold_id != f)[0]
                out.append(dict(protocol=proto, rep=r, fold=f, label=f"{proto}-r{r}f{f}", train_idx=tr, test_idx=te))
    # leave-one-element-out
    els = pd.Series([e for s in rec.sys_key for e in s.split('-')]).value_counts()
    for el, cnt in els.items():
        frac = cnt / n
        if cnt >= min_elem and frac <= max_frac:
            has = rec.sys_key.apply(lambda s: el in s.split('-')).values
            out.append(dict(protocol='element', rep=0, fold=0, label=f"element-{el}",
                            train_idx=np.where(~has)[0], test_idx=np.where(has)[0]))
    # temporal: train on literature up to 2018, test on 2019 papers
    yr = rec.year.values
    out.append(dict(protocol='temporal', rep=0, fold=0, label='temporal-2019',
                    train_idx=np.where(yr <= 2018)[0], test_idx=np.where(yr == 2019)[0]))
    return out


def inner_groups(rec_tr, protocol):
    """Grouping used for inner (tuning / calibration) folds; mirrors the deployment scenario."""
    if protocol == 'random':
        return None
    if protocol == 'composition':
        return rec_tr.comp_key.values
    return rec_tr.sys_key.values


def metrics(y, p):
    from scipy.stats import spearmanr
    y = np.asarray(y, float); p = np.asarray(p, float)
    e = p - y
    ss = ((y - y.mean()) ** 2).sum()
    return dict(n=len(y), MAE=np.abs(e).mean(), RMSE=np.sqrt((e ** 2).mean()),
                R2=1 - (e ** 2).sum() / ss if ss > 0 else np.nan,
                rho=spearmanr(y, p).correlation if len(y) > 2 else np.nan, bias=e.mean())


In [ ]:
"""Feature-set ablation on the frozen splits (fixed hyper-parameters, no tuning).
Models: XGB (fixed), GPR (ML-II kernel fit), kNN (k=5, distance-weighted).
Prints one line per (dataset, variant, model, split): A|ds|variant|model|label|protocol|n|MAE|RMSE|R2|rho
"""
import json
import sys
import time
import numpy as np
from sklearn.neighbors import KNeighborsRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from xgboost import XGBRegressor

VARIANTS = {
    'FRAC': ('FRAC', 'PROC', 'TEST'),
    'PHYS': ('PHYS', 'PROC', 'TEST'),
    'MAG': ('MAG', 'PROC', 'TEST'),
    'PHYS+MAG': ('PHYS', 'MAG', 'PROC', 'TEST'),
    'ALL': ('FRAC', 'PHYS', 'MAG', 'PROC', 'TEST'),
    'ALL-noPROC': ('FRAC', 'PHYS', 'MAG', 'TEST'),
    'ALL+PHASE': ('FRAC', 'PHYS', 'MAG', 'PROC', 'TEST', 'PHASE'),
}


def xgb_fixed(n_jobs):
    return XGBRegressor(n_estimators=500, max_depth=5, learning_rate=0.05, subsample=0.8, colsample_bytree=0.5,
                        min_child_weight=2, reg_lambda=1.0, tree_method='hist', n_jobs=n_jobs, random_state=0,
                        verbosity=0)


def run_ablation(load, feature_columns, Prep, metrics, data_dir, datasets=('HV', 'YS'), n_jobs=4, out=print):
    splits = json.load(open(f"{data_dir}/splits.json"))
    for ds in datasets:
        rec, Xdf = load(ds, data_dir)
        y = rec.target.values.astype(float)
        rid2i = {r: i for i, r in enumerate(rec.rid.values)}
        for vname, blocks in VARIANTS.items():
            cols = feature_columns(Xdf, blocks)
            X = Xdf[cols].values.astype(float)
            t0 = time.time()
            for sp in splits[ds]:
                te = np.array([rid2i[r] for r in sp['test']])
                mask = np.ones(len(y), bool); mask[te] = False
                tr = np.where(mask)[0]
                prep = Prep().fit(X[tr])
                Ztr, Zte = prep.transform(X[tr]), prep.transform(X[te])
                ym, ys = y[tr].mean(), y[tr].std()
                preds = {}
                m = xgb_fixed(n_jobs).fit(X[tr], y[tr]); preds['XGB'] = m.predict(X[te])
                k = ConstantKernel(1.0, (1e-2, 1e2)) * Matern(length_scale=10.0, length_scale_bounds=(1e-1, 1e3), nu=2.5) \
                    + WhiteKernel(0.1, (1e-4, 1.0))
                g = GaussianProcessRegressor(kernel=k, n_restarts_optimizer=1, random_state=0).fit(Ztr, (y[tr] - ym) / ys)
                preds['GPR'] = g.predict(Zte) * ys + ym
                kn = KNeighborsRegressor(n_neighbors=5, weights='distance').fit(Ztr, y[tr]); preds['kNN'] = kn.predict(Zte)
                for mn, p in preds.items():
                    mt = metrics(y[te], p)
                    out("A|" + "|".join([ds, vname, mn, sp['label'], sp['protocol'], str(mt['n'])] +
                                        [f"{mt[k]:.4f}" for k in ('MAE', 'RMSE', 'R2', 'rho')]))
            out(f"# {ds} {vname} n_features={len(cols)} done in {time.time()-t0:.0f}s")




In [ ]:
lines = []
def out(s):
    lines.append(s)
    if s.startswith('#'): print(s, flush=True)
t0 = time.time()
run_ablation(load, feature_columns, Prep, metrics, DATA, datasets=('HV', 'YS'), n_jobs=4, out=out)
open('/kaggle/working/ablation.txt', 'w').write('\n'.join(lines))
import pandas as pd
A = pd.DataFrame([l.split('|')[1:] for l in lines if l.startswith('A|')],
                 columns=['ds', 'variant', 'model', 'label', 'protocol', 'n', 'MAE', 'RMSE', 'R2', 'rho'])
for c in ['n', 'MAE', 'RMSE', 'R2', 'rho']: A[c] = A[c].astype(float)
S = A.groupby(['ds', 'variant', 'model', 'protocol']).agg(RMSE=('RMSE', 'mean'), RMSE_sd=('RMSE', 'std'), R2=('R2', 'mean'), R2_sd=('R2', 'std'), rho=('rho', 'mean'), k=('RMSE', 'size')).reset_index()
print('=====ABL_BEGIN=====')
print(S.to_csv(index=False, float_format='%.4f'))
print('=====ABL_END=====')
print('ABL_MIN', round((time.time() - t0) / 60, 1))

In [ ]:
"""Sensitivity: does de-duplication alone remove the optimism of random CV?
Hardness records are aggregated to one record per (composition, processing) (median HV), then evaluated with
random and system-grouped 5-fold CV (3 repeats) using fixed-hyper-parameter XGB and GPR."""
import numpy as np
import pandas as pd
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel


def main():
    rec, X = load('HV', DATA)
    cols = feature_columns(X, DEFAULT_BLOCKS)
    df = pd.concat([rec.reset_index(drop=True), X[cols].reset_index(drop=True)], axis=1)
    agg = df.groupby(['comp_key', 'proc'], as_index=False).agg({**{c: 'first' for c in cols}, 'target': 'median', 'sys_key': 'first'})
    out = []
    for tag, d in [('original', df), ('deduplicated', agg)]:
        Xv, y = d[cols].values.astype(float), d.target.values.astype(float)
        for proto in ('random', 'system'):
            for r in range(3):
                rng = np.random.default_rng(2026 + r)
                if proto == 'random':
                    f = np.empty(len(y), int); f[rng.permutation(len(y))] = np.arange(len(y)) % 5
                else:
                    f = balanced_group_folds(d.sys_key.values, 5, rng)
                pred = {'XGB': np.zeros(len(y)), 'GPR': np.zeros(len(y))}
                for k in range(5):
                    tr, te = f != k, f == k
                    pred['XGB'][te] = xgb_fixed(2).fit(Xv[tr], y[tr]).predict(Xv[te])
                    p = Prep().fit(Xv[tr]); ym, ys = y[tr].mean(), y[tr].std()
                    kern = ConstantKernel(1.0, (1e-2, 1e2)) * Matern(10.0, (1e-1, 1e3), nu=2.5) + WhiteKernel(0.1, (1e-4, 1.0))
                    g = GaussianProcessRegressor(kern, n_restarts_optimizer=1, random_state=0).fit(p.transform(Xv[tr]), (y[tr] - ym) / ys)
                    pred['GPR'][te] = g.predict(p.transform(Xv[te])) * ys + ym
                for m, pv in pred.items():
                    mt = metrics(y, pv)
                    out.append(dict(data=tag, n=len(y), protocol=proto, rep=r, model=m, RMSE=mt['RMSE'], R2=mt['R2']))
    res = pd.DataFrame(out)
    s = res.groupby(['data', 'n', 'protocol', 'model'])[['RMSE', 'R2']].agg(['mean', 'std']).round(3)
    print(s)
    res.to_csv('/kaggle/working/dedup_sensitivity.csv', index=False)
    print('=====DEDUP_BEGIN=====')
    print(res.to_csv(index=False))
    print('=====DEDUP_END=====')



main()
